# Wing Model IV Calibration and Delta-Hedged Backtest

In [ ]:
from __future__ import annotations

import ast
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

In [ ]:
def find_project_root(start_paths: Optional[Sequence[Path]] = None) -> Path:
    candidates = []

    if start_paths is None:
        start_paths = []

    for p in start_paths:
        if p is None:
            continue
        p = Path(p).resolve()
        candidates.extend([p, *p.parents])

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    ordered = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            ordered.append(c)

    for root in ordered:
        if (root / "data" / "test_options_data_with_iv.csv").exists():
            return root

    raise FileNotFoundError(
        "Could not find the project root. Please make sure "
        "'data/test_options_data_with_iv.csv' exists."
    )

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "test_options_data_with_iv.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data path    :", DATA_PATH)
print("Figures dir  :", FIGURES_DIR)

def savefig_local(filename: str, dpi: int = 200, **kwargs) -> Path:
    path = FIGURES_DIR / filename
    plt.savefig(path, dpi=dpi, **kwargs)
    print(f"Saved figure: {path}")
    return path

In [ ]:
PARAMS = {
    "mny_low": 0.90,
    "mny_high": 1.10,
    "z_enter": 2.0,
    "z_exit": 0.5,
    "max_trades_per_expiry_per_day": 1,
    "max_hold_days": 10,
    "min_dte_exit": 3,
    "contracts_per_trade": 1,
    "min_volume": 1,
    "min_price": 0.01,
    "initial_cash": 1_000_000.0,
    "multiplier": 100,
    "delta_band": 0.0,
    "opt_commission_per_contract": 0.50,
    "opt_eta": 0.0000,
    "und_bps": 1.0,
    "und_fee_per_share": 0.0,
    "example_row_index": 2000,
}

In [ ]:
def safe_list(x) -> List[float]:
    if pd.isna(x):
        return []
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)

    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []

    s2 = (
        s.replace("nan", "None")
        .replace("NaN", "None")
        .replace("NAN", "None")
        .replace("inf", "None")
        .replace("Inf", "None")
        .replace("-inf", "None")
        .replace("-Inf", "None")
    )

    try:
        out = ast.literal_eval(s2)
    except Exception:
        return []

    if not isinstance(out, (list, tuple)):
        return []

    return [np.nan if v is None else v for v in out]

def load_dataset(data_path: Path = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"])
    df["expiration_date"] = pd.to_datetime(df["expiration_date"])
    return df

def choose_example_row_index(df: pd.DataFrame, requested_index: int) -> int:
    if df.empty:
        raise ValueError("The dataset is empty.")
    return int(np.clip(requested_index, 0, len(df) - 1))

def design_matrix_quadratic(K: np.ndarray) -> np.ndarray:
    K = np.asarray(K, dtype=float)
    return np.column_stack([np.ones(len(K)), K, K * K])

def fit_quadratic_ols(K: np.ndarray, y: np.ndarray) -> np.ndarray:
    X = design_matrix_quadratic(K)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef

def fit_quadratic_wls(K: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    K = np.asarray(K, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    if np.sum(w) <= 0:
        w = np.ones_like(w, dtype=float)
    w = w / np.sum(w)

    X = design_matrix_quadratic(K)
    sw = np.sqrt(w)
    coef, *_ = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)
    return coef

def robust_sigma(residuals: np.ndarray) -> float:
    r = np.asarray(residuals, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return 1e-6
    med = np.median(r)
    mad = np.median(np.abs(r - med))
    sig = 1.4826 * mad
    if (not np.isfinite(sig)) or sig < 1e-8:
        sig = np.std(r)
    if (not np.isfinite(sig)) or sig < 1e-8:
        sig = 1e-6
    return float(sig)

def bs_delta(S: float, K: float, r: float, T: float, iv: float, opt_type: str) -> float:
    if (S <= 0) or (K <= 0) or (T <= 0) or (iv <= 0) or (not np.isfinite(iv)):
        return 0.0
    d1 = (np.log(S / K) + (r + 0.5 * iv * iv) * T) / (iv * np.sqrt(T))
    if opt_type == "C":
        return float(norm.cdf(d1))
    return float(norm.cdf(d1) - 1.0)

def option_trade_cost_simple(price: float, qty_contracts: float, params: Dict) -> float:
    q = float(abs(qty_contracts))
    p = float(max(price, 0.0))
    M = params["multiplier"]
    c_opt = params["opt_commission_per_contract"]
    eta = params["opt_eta"]
    return q * (c_opt + eta * M * p)

def underlying_trade_cost_simple(S: float, shares_change: float, params: Dict) -> float:
    S = float(max(S, 0.0))
    sh = float(abs(shares_change))
    bps = params["und_bps"]
    c_und = params["und_fee_per_share"]
    return sh * S * (bps / 10000.0) + sh * c_und

def max_affordable_contracts(cash: float, price: float, params: Dict) -> int:
    per_contract_outflow = price * params["multiplier"] + option_trade_cost_simple(price, 1, params)
    if per_contract_outflow <= 0:
        return 0
    return int(cash // per_contract_outflow)

def max_affordable_shares(cash: float, S: float, params: Dict) -> float:
    per_share_outflow = S * (1.0 + params["und_bps"] / 10000.0) + params["und_fee_per_share"]
    if per_share_outflow <= 0:
        return 0.0
    return float(cash // per_share_outflow)

## 1. Load the local CSV

In [ ]:
df = load_dataset(DATA_PATH)
print(df.shape)
df.head()

## 2. Single-row OLS / WLS smile fit example

In [ ]:
def extract_row_arrays(row: pd.Series) -> Optional[Dict[str, np.ndarray]]:
    S0 = float(row["S0"])
    r = float(row["r"])
    T = float(row["DTE"]) / 365.0

    K_raw = np.asarray(safe_list(row.get("K")), dtype=float)
    c_last = np.asarray(safe_list(row.get("c_last")), dtype=float)
    p_last = np.asarray(safe_list(row.get("p_last")), dtype=float)
    call_iv_raw = np.asarray(safe_list(row.get("call_iv")), dtype=float)
    put_iv_raw = np.asarray(safe_list(row.get("put_iv")), dtype=float)
    c_vol = np.asarray(safe_list(row.get("c_volume")), dtype=float)
    p_vol = np.asarray(safe_list(row.get("p_volume")), dtype=float)

    lens = [len(K_raw), len(c_last), len(p_last), len(call_iv_raw), len(put_iv_raw), len(c_vol), len(p_vol)]
    L = min(lens) if lens else 0
    if L <= 0:
        return None

    return {
        "S0": S0,
        "r": r,
        "T": T,
        "K": K_raw[:L],
        "c_last": c_last[:L],
        "p_last": p_last[:L],
        "call_iv": call_iv_raw[:L],
        "put_iv": put_iv_raw[:L],
        "c_volume": c_vol[:L],
        "p_volume": p_vol[:L],
    }

def build_otm_smile_slice(row: pd.Series, mny_low: float = 0.90, mny_high: float = 1.10):
    arrays = extract_row_arrays(row)
    if arrays is None:
        return None

    S0 = arrays["S0"]
    K_raw = arrays["K"]
    mask = (K_raw >= S0 * mny_low) & (K_raw <= S0 * mny_high)

    K = K_raw[mask]
    call_iv = arrays["call_iv"][mask]
    put_iv = arrays["put_iv"][mask]
    c_volume = arrays["c_volume"][mask]
    p_volume = arrays["p_volume"][mask]

    if len(K) < 5:
        return None

    is_otm_put = K <= S0
    is_otm_call = K > S0

    otm_iv = np.zeros(len(K))
    otm_iv[is_otm_put] = np.where(np.isfinite(put_iv[is_otm_put]), put_iv[is_otm_put], 0.0)
    otm_iv[is_otm_call] = np.where(np.isfinite(call_iv[is_otm_call]), call_iv[is_otm_call], 0.0)

    otm_volume = np.zeros(len(K))
    otm_volume[is_otm_put] = np.where(np.isfinite(p_volume[is_otm_put]), p_volume[is_otm_put], 0.0)
    otm_volume[is_otm_call] = np.where(np.isfinite(c_volume[is_otm_call]), c_volume[is_otm_call], 0.0)

    valid = np.isfinite(otm_iv) & (otm_iv > 0) & np.isfinite(K)
    K = K[valid]
    otm_iv = otm_iv[valid]
    otm_volume = otm_volume[valid]

    if len(K) < 5:
        return None

    weights = otm_volume.astype(float)
    if np.nansum(weights) <= 0:
        weights = np.ones_like(K, dtype=float)
    weights = weights / np.nansum(weights)

    return {"S0": S0, "K": K, "otm_iv": otm_iv, "weights": weights}

def run_single_row_diagnostics(df: pd.DataFrame, params: Dict, row_index: Optional[int] = None):
    row_index = choose_example_row_index(df, params["example_row_index"] if row_index is None else row_index)
    row = df.iloc[row_index]
    smile = build_otm_smile_slice(row, params["mny_low"], params["mny_high"])
    if smile is None:
        raise ValueError(f"Row {row_index} does not contain enough valid OTM points.")

    K = smile["K"]
    y = smile["otm_iv"]
    w = smile["weights"]

    ols_coef = fit_quadratic_ols(K, y)
    wls_coef = fit_quadratic_wls(K, y, w)

    y_ols = design_matrix_quadratic(K) @ ols_coef
    y_wls = design_matrix_quadratic(K) @ wls_coef

    plt.figure(figsize=(10, 4))
    plt.plot(K, y, "o-", label="OTM Market IV")
    plt.plot(K, y_ols, "-", label="Wing Model (OLS)")
    plt.plot(K, y_wls, "-", label="Wing Model (WLS)")
    plt.xlabel("Strike")
    plt.ylabel("Implied Volatility")
    plt.title(f"Single-Row Wing Model Fit Example (row={row_index})")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {"row_index": row_index, "K": K, "otm_iv": y, "ols_coef": ols_coef, "wls_coef": wls_coef}

diag = run_single_row_diagnostics(df, PARAMS)
diag["ols_coef"], diag["wls_coef"]

## 3. Build long-format option panels and calibrate z-scores

In [ ]:
def build_option_panels(df: pd.DataFrame, params: Dict):
    opt_cols = ["date", "exp", "S", "r", "DTE", "T", "K", "type", "price", "iv", "volume"]
    rows = []

    for _, row in df.iterrows():
        date = row["Date"]
        exp = row["expiration_date"]
        S0 = float(row["S0"])
        r = float(row["r"])
        dte = float(row["DTE"])
        T = dte / 365.0

        K_list = safe_list(row.get("K"))
        c_last = safe_list(row.get("c_last"))
        p_last = safe_list(row.get("p_last"))
        c_iv = safe_list(row.get("call_iv"))
        p_iv = safe_list(row.get("put_iv"))
        c_vol = safe_list(row.get("c_volume"))
        p_vol = safe_list(row.get("p_volume"))

        lens = [len(K_list), len(c_last), len(p_last), len(c_iv), len(p_iv), len(c_vol), len(p_vol)]
        L = min(lens) if lens else 0
        if L <= 0:
            continue

        K_arr = np.asarray(K_list[:L], dtype=float)
        c_last_arr = np.asarray(c_last[:L], dtype=float)
        p_last_arr = np.asarray(p_last[:L], dtype=float)
        c_iv_arr = np.asarray(c_iv[:L], dtype=float)
        p_iv_arr = np.asarray(p_iv[:L], dtype=float)
        c_vol_arr = np.asarray(c_vol[:L], dtype=float)
        p_vol_arr = np.asarray(p_vol[:L], dtype=float)

        for k, cl, pl, civ, piv, cv, pv in zip(K_arr, c_last_arr, p_last_arr, c_iv_arr, p_iv_arr, c_vol_arr, p_vol_arr):
            rows.append({"date": date, "exp": exp, "S": S0, "r": r, "DTE": dte, "T": T,
                         "K": float(k), "type": "C", "price": float(cl), "iv": float(civ), "volume": float(cv)})
            rows.append({"date": date, "exp": exp, "S": S0, "r": r, "DTE": dte, "T": T,
                         "K": float(k), "type": "P", "price": float(pl), "iv": float(piv), "volume": float(pv)})

    opt_all = pd.DataFrame(rows, columns=opt_cols)
    if opt_all.empty:
        raise ValueError("No valid option rows created. Check list parsing and CSV fields.")

    opt_all = opt_all.replace([np.inf, -np.inf], np.nan).dropna(subset=opt_cols)
    opt_all = opt_all[(opt_all["iv"] > 0) & (opt_all["price"] >= params["min_price"])]

    opt_all["is_otm"] = (
        ((opt_all["type"] == "P") & (opt_all["K"] <= opt_all["S"]))
        | ((opt_all["type"] == "C") & (opt_all["K"] > opt_all["S"]))
    )

    opt_sig = opt_all[
        (opt_all["K"] >= opt_all["S"] * params["mny_low"])
        & (opt_all["K"] <= opt_all["S"] * params["mny_high"])
    ].copy()

    return opt_all.reset_index(drop=True), opt_sig.reset_index(drop=True)

def calibrate_signals(opt_sig: pd.DataFrame) -> pd.DataFrame:
    opt_sig = opt_sig.copy()
    opt_sig["iv_fit"] = np.nan
    opt_sig["z"] = np.nan

    for (d, e), g in opt_sig.groupby(["date", "exp"], sort=False):
        g_otm = g[g["is_otm"]].copy()
        if len(g_otm) < 5:
            continue

        K = g_otm["K"].values
        y = g_otm["iv"].values
        w = np.maximum(g_otm["volume"].values, 0.0)

        coef = fit_quadratic_wls(K, y, w)

        iv_fit_all = design_matrix_quadratic(g["K"].values) @ coef
        res_otm = y - (design_matrix_quadratic(K) @ coef)
        sig = robust_sigma(res_otm)

        idx = g.index
        opt_sig.loc[idx, "iv_fit"] = iv_fit_all
        opt_sig.loc[idx, "z"] = (g["iv"].values - iv_fit_all) / sig

    opt_sig = opt_sig.dropna(subset=["iv_fit", "z"]).reset_index(drop=True)
    if opt_sig.empty:
        raise ValueError("No fitted rows after calibration.")
    return opt_sig

opt_all, opt_sig = build_option_panels(df, PARAMS)
opt_sig = calibrate_signals(opt_sig)

print("opt_all:", opt_all.shape)
print("opt_sig:", opt_sig.shape)
opt_sig[["date", "exp", "K", "type", "iv", "iv_fit", "z"]].head()

## 4. Run the delta-hedged backtest



In [ ]:
def run_backtest(opt_all: pd.DataFrame, opt_sig: pd.DataFrame, params: Dict):
    positions = {}
    cash = float(params["initial_cash"])
    hedge_shares = 0.0

    opt_all_map = {(r["date"], r["exp"], float(r["K"]), r["type"]): idx for idx, r in opt_all.iterrows()}
    opt_sig_map = {(r["date"], r["exp"], float(r["K"]), r["type"]): idx for idx, r in opt_sig.iterrows()}

    def get_row_all(date, exp, K, typ):
        idx = opt_all_map.get((date, exp, float(K), typ), None)
        return None if idx is None else opt_all.loc[idx]

    def get_row_sig(date, exp, K, typ):
        idx = opt_sig_map.get((date, exp, float(K), typ), None)
        return None if idx is None else opt_sig.loc[idx]

    dates = sorted(opt_all["date"].unique())
    pending_trades = []
    daily_log = []

    for t, date in enumerate(dates):
        day_data = opt_sig[opt_sig["date"] == date].copy()
        day_data_all = opt_all[opt_all["date"] == date].copy()
        if day_data_all.empty:
            continue

        S_today = float(day_data_all["S"].median())

        expired_keys = []
        for (exp, K, typ), pos in positions.items():
            if pd.to_datetime(date) >= pd.to_datetime(exp):
                intrinsic = max(S_today - float(K), 0.0) if typ == "C" else max(float(K) - S_today, 0.0)
                cash += pos["qty"] * params["multiplier"] * intrinsic
                expired_keys.append((exp, K, typ))

        for key in expired_keys:
            del positions[key]

        for key, pos in positions.items():
            exp, K, typ = key
            pos["days_held"] += 1

            row_now_all = get_row_all(date, exp, K, typ)
            if row_now_all is not None:
                pos["price"] = float(row_now_all["price"])
                pos["iv"] = float(row_now_all["iv"])
                pos["DTE"] = float(row_now_all["DTE"])
                pos["T"] = float(row_now_all["T"])
                pos["S"] = float(row_now_all["S"])
                pos["r"] = float(row_now_all["r"])
                pos["volume"] = float(row_now_all["volume"])

            row_now_sig = get_row_sig(date, exp, K, typ)
            pos["z"] = float(row_now_sig["z"]) if row_now_sig is not None else np.nan

        carry_over = []
        for (exp, K, typ, trade_qty) in pending_trades:
            if pd.to_datetime(date) >= pd.to_datetime(exp):
                continue

            row_now_all = get_row_all(date, exp, K, typ)
            if row_now_all is None:
                carry_over.append((exp, K, typ, trade_qty))
                continue

            price = float(row_now_all["price"])
            key = (exp, float(K), typ)
            trade_qty_exec = float(trade_qty)

            if trade_qty_exec > 0:
                is_buy_to_close_short = (key in positions) and (positions[key]["qty"] < 0)
                max_q = max_affordable_contracts(cash, price, params)
                if max_q <= 0:
                    if is_buy_to_close_short:
                        carry_over.append((exp, K, typ, trade_qty))
                    continue
                if trade_qty_exec > max_q:
                    trade_qty_exec = float(max_q)

            cash -= trade_qty_exec * price * params["multiplier"]
            cash -= option_trade_cost_simple(price, trade_qty_exec, params)

            if key not in positions:
                row_now_sig = get_row_sig(date, exp, K, typ)
                z_now = float(row_now_sig["z"]) if row_now_sig is not None else np.nan
                positions[key] = {
                    "qty": float(trade_qty_exec),
                    "price": price,
                    "iv": float(row_now_all["iv"]),
                    "z": z_now,
                    "DTE": float(row_now_all["DTE"]),
                    "T": float(row_now_all["T"]),
                    "S": float(row_now_all["S"]),
                    "r": float(row_now_all["r"]),
                    "volume": float(row_now_all["volume"]),
                    "days_held": 0,
                }
            else:
                positions[key]["qty"] += float(trade_qty_exec)
                positions[key]["price"] = price
                if abs(positions[key]["qty"]) < 1e-12:
                    del positions[key]

        pending_trades = carry_over

        total_delta_shares = 0.0
        for (exp, K, typ), pos in positions.items():
            dlt = bs_delta(float(pos.get("S", S_today)), float(K), float(pos.get("r", 0.0)),
                           float(pos.get("T", 0.0)), float(pos.get("iv", 0.0)), typ)
            total_delta_shares += pos["qty"] * params["multiplier"] * dlt

        target_hedge = -total_delta_shares
        shares_change = target_hedge - hedge_shares

        if abs(shares_change) > params["delta_band"]:
            shares_exec = float(shares_change)

            if shares_exec > 0:
                max_sh = max_affordable_shares(cash, S_today, params)
                shares_exec = 0.0 if max_sh <= 0 else min(shares_exec, max_sh)

            if abs(shares_exec) > 0:
                cash -= shares_exec * S_today
                cash -= underlying_trade_cost_simple(S_today, shares_exec, params)
                hedge_shares += shares_exec

        option_value = 0.0
        gross_option_notional = 0.0
        for pos in positions.values():
            option_value += pos["qty"] * params["multiplier"] * pos["price"]
            gross_option_notional += abs(pos["qty"]) * params["multiplier"] * pos["price"]

        underlying_value = hedge_shares * S_today
        gross_underlying_notional = abs(hedge_shares) * S_today
        equity = cash + option_value + underlying_value

        daily_log.append({
            "date": pd.to_datetime(date).date(),
            "equity": equity,
            "cash": cash,
            "option_value": option_value,
            "underlying_value": underlying_value,
            "hedge_shares": hedge_shares,
            "n_positions": len(positions),
            "gross_option_notional": gross_option_notional,
            "gross_underlying_notional": gross_underlying_notional,
        })

        if t == len(dates) - 1:
            continue

        for key, pos in list(positions.items()):
            exp, K, typ = key
            z = pos.get("z", np.nan)
            dte = pos.get("DTE", np.nan)

            exit_by_revert = np.isfinite(z) and (abs(z) <= params["z_exit"])
            exit_by_time = pos.get("days_held", 0) >= params["max_hold_days"]
            exit_by_dte = np.isfinite(dte) and (dte <= params["min_dte_exit"])

            if exit_by_revert or exit_by_time or exit_by_dte:
                pending_trades.append((exp, float(K), typ, -float(pos["qty"])))

        for exp, g in day_data.groupby("exp"):
            g_otm = g[g["is_otm"]].copy()
            g_otm = g_otm[(g_otm["volume"] >= params["min_volume"]) & (g_otm["price"] >= params["min_price"])]
            if g_otm.empty:
                continue

            underv = g_otm[g_otm["z"] <= -params["z_enter"]].sort_values("z")
            if not underv.empty:
                for _, row in underv.head(params["max_trades_per_expiry_per_day"]).iterrows():
                    key = (row["exp"], float(row["K"]), row["type"])
                    if key not in positions:
                        pending_trades.append((row["exp"], float(row["K"]), row["type"], +params["contracts_per_trade"]))

            over = g_otm[g_otm["z"] >= params["z_enter"]].sort_values("z", ascending=False)
            if not over.empty:
                for _, row in over.head(params["max_trades_per_expiry_per_day"]).iterrows():
                    key = (row["exp"], float(row["K"]), row["type"])
                    if key not in positions:
                        pending_trades.append((row["exp"], float(row["K"]), row["type"], -params["contracts_per_trade"]))

    log_df = pd.DataFrame(daily_log)
    log_df["date"] = pd.to_datetime(log_df["date"])
    log_df = log_df.sort_values("date").reset_index(drop=True)

    log_df["ret"] = log_df["equity"].pct_change().fillna(0.0)
    log_df["cum_ret"] = log_df["equity"] / log_df["equity"].iloc[0] - 1.0
    log_df["cummax"] = log_df["equity"].cummax()
    log_df["dd"] = log_df["equity"] / log_df["cummax"] - 1.0

    mdd = float(log_df["dd"].min())
    trough_idx = int(log_df["dd"].idxmin())
    peak_idx = int(log_df["equity"][: trough_idx + 1].idxmax())
    peak_date = log_df.loc[peak_idx, "date"]
    trough_date = log_df.loc[trough_idx, "date"]

    ret_mean = log_df["ret"].mean()
    ret_std = log_df["ret"].std()
    sharpe = np.nan if ret_std <= 1e-12 else (ret_mean / ret_std) * np.sqrt(252)

    summary = {
        "start_equity": float(log_df["equity"].iloc[0]),
        "end_equity": float(log_df["equity"].iloc[-1]),
        "total_return_pct": float((log_df["equity"].iloc[-1] / log_df["equity"].iloc[0] - 1.0) * 100.0),
        "annualized_sharpe": None if not np.isfinite(sharpe) else float(sharpe),
        "max_drawdown_pct": float(mdd * 100.0),
        "mdd_peak_date": peak_date.date().isoformat(),
        "mdd_trough_date": trough_date.date().isoformat(),
    }
    return log_df, summary

log_df, summary = run_backtest(opt_all, opt_sig, PARAMS)
summary

## 5. Save the report figures to `figures/`

In [ ]:
def plot_equity_curve(log_df: pd.DataFrame):
    plt.figure(figsize=(10, 4))
    plt.plot(log_df["date"], log_df["equity"], label="Equity")
    plt.title("Equity Curve (Option Entry/Exit Costs + Daily Hedge Costs; 1-day Signal Lag)")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    savefig_local("equity_curve.png")
    plt.show()

def plot_exposure_chart(log_df: pd.DataFrame):
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(log_df["date"], log_df["gross_option_notional"], label="Gross Option Notional")
    ax1.plot(log_df["date"], log_df["gross_underlying_notional"], label="Gross Underlying Notional")
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Gross Notional (USD)")
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(log_df["date"], log_df["n_positions"], linestyle="--", label="Number of Positions")
    ax2.set_ylabel("n_positions")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.title("Portfolio Exposure Over Time")
    plt.tight_layout()
    savefig_local("exposure_chart.png")
    plt.show()

def plot_cumulative_return_with_mdd(log_df: pd.DataFrame):
    mdd = float(log_df["dd"].min())
    trough_idx = int(log_df["dd"].idxmin())
    peak_idx = int(log_df["equity"][: trough_idx + 1].idxmax())
    peak_date = log_df.loc[peak_idx, "date"]
    trough_date = log_df.loc[trough_idx, "date"]

    plt.figure(figsize=(10, 4))
    plt.plot(log_df["date"], log_df["cum_ret"], label="Cumulative Return")
    plt.scatter(
        [peak_date, trough_date],
        [log_df.loc[peak_idx, "cum_ret"], log_df.loc[trough_idx, "cum_ret"]],
        zorder=3,
        label="MDD Peak/Trough",
    )
    plt.plot(
        [peak_date, trough_date],
        [log_df.loc[peak_idx, "cum_ret"], log_df.loc[trough_idx, "cum_ret"]],
        linewidth=1.5,
        label="MDD Segment",
    )
    plt.text(
        trough_date,
        log_df.loc[trough_idx, "cum_ret"],
        f" MDD = {mdd * 100:.2f}%\n Peak={peak_date.date()}\n Trough={trough_date.date()}",
        va="top",
    )
    plt.title("Cumulative Return with Max Drawdown (MDD) Annotation")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    savefig_local("cum_return_mdd.png")
    plt.show()

def plot_mispricing_example(opt_sig: pd.DataFrame, params: Dict):
    example = None
    for (d, e), g in opt_sig.groupby(["date", "exp"], sort=True):
        g_otm = g[g["is_otm"]].copy()
        if g_otm.empty:
            continue
        flagged = g_otm[np.abs(g_otm["z"]) >= params["z_enter"]]
        if len(flagged) > 0:
            example = (d, e, g_otm.copy(), flagged.copy())
            break

    if example is None:
        print("No mispricing example found with the current z_enter threshold.")
        return

    d_ex, e_ex, g_otm, flagged = example
    coef = fit_quadratic_wls(g_otm["K"].values, g_otm["iv"].values, np.maximum(g_otm["volume"].values, 0.0))

    K_grid = np.linspace(np.min(g_otm["K"].values), np.max(g_otm["K"].values), 200)
    iv_grid = design_matrix_quadratic(K_grid) @ coef

    plt.figure(figsize=(10, 4))
    plt.scatter(g_otm["K"], g_otm["iv"], label="OTM Market IV", alpha=0.8)
    plt.plot(K_grid, iv_grid, label="Fitted IV Curve", linewidth=2)
    plt.scatter(flagged["K"], flagged["iv"], label="Mispriced (|z|>=z_enter)", zorder=4)

    for _, rr in flagged.iterrows():
        plt.text(rr["K"], rr["iv"], f" {rr['type']}", fontsize=9, va="bottom")

    plt.title(
        "Mispricing Detection Example  "
        f"Date={pd.to_datetime(d_ex).date()}  "
        f"Exp={pd.to_datetime(e_ex).date()}"
    )
    plt.xlabel("Strike K")
    plt.ylabel("Implied Volatility")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    savefig_local("mispricing_example.png")
    plt.show()

plot_equity_curve(log_df)
plot_exposure_chart(log_df)
plot_cumulative_return_with_mdd(log_df)
plot_mispricing_example(opt_sig, PARAMS)